# LeetCode 3806 — Maximum Bitwise AND After Increment Operations bằng Python

- Link: https://leetcode.com/problems/maximum-bitwise-and-after-increment-operations/
- Độ khó: **Hard**
- Chủ đề: Thao tác bit, tham lam dựng đáp án từ bit cao xuống, "số nhỏ nhất ≥ x chứa mọi bit của T", sắp xếp / chọn `m` phần tử rẻ nhất

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumAND(self, nums: List[int], k: int, m: int) -> int:
```

## 1. Mục tiêu học

1. Hiểu khuôn **dựng đáp án theo bit từ cao xuống**: thử bật bit, giữ lại nếu vẫn khả thi.
2. Biết điều kiện "AND của nhóm có mọi bit của `T`" ⇔ **mỗi phần tử chứa `T`** (`y & T == T`).
3. Tính nhanh **chi phí tối thiểu** để tăng `x` lên số nhỏ nhất `y ≥ x` chứa `T`: tìm bit cao nhất mà `T` có nhưng `x` thiếu, thay các bit thấp hơn bằng bit của `T`.
4. Kiểm tra khả thi: tổng `m` chi phí nhỏ nhất `≤ k`.
5. Kiểm chứng công thức chi phí và thuật toán bằng vét cạn.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums`, số `k` và `m`. Được thực hiện **tối đa `k`** thao tác, mỗi thao tác tăng một phần tử lên 1. Sau đó chọn **một nhóm `m` phần tử**; tìm giá trị **AND bit** lớn nhất có thể của nhóm đó.

**Ví dụ 1:** `nums = [3,1,2], k = 8, m = 2` → `6` (đưa `3 → 6`, `2 → 6`, tốn 7).

**Ví dụ 2:** `nums = [1,2,8,4], k = 7, m = 3` → `4` (`1 → 4`, `2 → 4`, giữ `4`, tốn 5).

**Ví dụ 3:** `nums = [1,1], k = 3, m = 2` → `2`.

**Ràng buộc:** `1 ≤ n ≤ 5·10^4`, `1 ≤ nums[i], k ≤ 10^9`, `1 ≤ m ≤ n`.

In [1]:
from typing import List
from itertools import combinations, product
import heapq, random, time

examples = [([3, 1, 2], 8, 2, 6), ([1, 2, 8, 4], 7, 3, 4), ([1, 1], 3, 2, 2)]
for nums, k, m, exp in examples:
    print(nums, "k =", k, "m =", m, "-> kỳ vọng", exp)

[3, 1, 2] k = 8 m = 2 -> kỳ vọng 6
[1, 2, 8, 4] k = 7 m = 3 -> kỳ vọng 4
[1, 1] k = 3 m = 2 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. "Chứa mọi bit của `T`" và AND

`y ⊇ T` (theo bit) ⇔ `y & T == T`. AND của một nhóm chứa `T` ⇔ **mọi** phần tử của nhóm chứa `T`. Vì vậy "AND ≥ T theo nghĩa bit" là điều kiện **từng phần tử** — tách được.

In [2]:
T = 0b110
for y in (6, 7, 14, 5, 4):
    print(y, bin(y), "chứa T?", y & T == T)
print("AND(6, 7, 14) =", 6 & 7 & 14, "chứa T:", (6 & 7 & 14) & T == T)

6 0b110 chứa T? True
7 0b111 chứa T? True
14 0b1110 chứa T? True
5 0b101 chứa T? False
4 0b100 chứa T? False
AND(6, 7, 14) = 6 chứa T: True


### 3.2. Số nhỏ nhất `y ≥ x` chứa `T`

- Nếu `x` đã chứa `T`: `y = x`.
- Ngược lại, gọi `j − 1` là **bit cao nhất** mà `T` có còn `x` không có: `j = (T & ~x).bit_length()`. Các bit **cao hơn** `j − 1` của `x` đã chứa phần tương ứng của `T`. Ta phải bật bit `j − 1` (làm số tăng), nên phần thấp hơn nên **nhỏ nhất có thể** mà vẫn chứa `T`, tức đúng bằng các bit của `T`:

`y = (x với j bit thấp xóa đi) | (T & (2^j − 1))`, chi phí `y − x = (T & mask) − (x & mask)`, `mask = 2^j − 1`.

In [3]:
def min_cost(x, T):
    j = (T & ~x).bit_length()
    mask = (1 << j) - 1
    return (T & mask) - (x & mask)

def min_cost_brute(x, T):
    y = x
    while y & T != T:
        y += 1
    return y - x

for x, T in [(3, 6), (2, 6), (1, 4), (5, 6), (8, 4), (13, 10)]:
    print(f"x={x:2} ({x:05b}) T={T:2} ({T:05b}): công thức {min_cost(x, T)}, duyệt {min_cost_brute(x, T)}")

x= 3 (00011) T= 6 (00110): công thức 3, duyệt 3
x= 2 (00010) T= 6 (00110): công thức 4, duyệt 4
x= 1 (00001) T= 4 (00100): công thức 3, duyệt 3
x= 5 (00101) T= 6 (00110): công thức 1, duyệt 1
x= 8 (01000) T= 4 (00100): công thức 4, duyệt 4
x=13 (01101) T=10 (01010): công thức 1, duyệt 1


### 3.3. Tổng `m` giá trị nhỏ nhất

`sorted(a)[:m]` (`O(n log n)`) hoặc `heapq.nsmallest(m, a)` (`O(n log m)`).

In [4]:
a = [7, 3, 9, 1, 4]
print(sum(sorted(a)[:3]), sum(heapq.nsmallest(3, a)))

8 8


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Muốn AND lớn ⇒ ưu tiên **bit cao**. Thử từ bit cao nhất: "có thể có AND chứa bit này không?". Nếu được thì **giữ** (vì một bit cao lớn hơn tổng mọi bit thấp), rồi thử bit kế tiếp **cùng với** các bit đã giữ.

Tính tay ví dụ 1 (`[3, 1, 2]`, `k = 8`, `m = 2`):

| thử T | chi phí từng số (3, 1, 2) | 2 rẻ nhất | ≤ 8? | giữ |
|---|---|---|---|---|
| 8 (1000) | 5, 7, 6 | 11 | ✘ | 0 |
| 4 (0100) | 1, 3, 2 | 3 | ✔ | 4 |
| 6 (0110) | 3, 5, 4 | 7 | ✔ | 6 |
| 7 (0111) | 4, 6, 5 | 9 | ✘ | 6 |

Đáp án `6` ✔.

In [5]:
for T in (8, 4, 6, 7):
    c = [min_cost(x, T) for x in [3, 1, 2]]
    print(T, c, sum(sorted(c)[:2]))

8 [5, 7, 6] 11
4 [1, 3, 2] 3
6 [3, 5, 4] 7
7 [4, 6, 5] 9


## 5. Cách 1 — Brute Force: thử mọi mức tăng (dữ liệu nhỏ)

Với mỗi bộ số lần tăng `(t_0, …, t_{n−1})` có tổng `≤ k`, xét mọi nhóm `m` phần tử và lấy AND lớn nhất. Hàm mũ — chỉ dùng cho `n, k` rất nhỏ.

In [6]:
def brute(nums, k, m):
    n = len(nums); best = 0
    for t in product(range(k + 1), repeat=n):
        if sum(t) > k: continue
        vals = [x + d for x, d in zip(nums, t)]
        for S in combinations(vals, m):
            a = S[0]
            for v in S[1:]: a &= v
            best = max(best, a)
    return best

for nums, k, m, exp in examples:
    if len(nums) <= 3:
        print(brute(nums, k, m), "kỳ vọng", exp)

6 kỳ vọng 6
2 kỳ vọng 2


**Nút thắt:** số bộ `t` khổng lồ. Mục 3 cho thấy với mỗi "đích" `T`, chi phí từng phần tử **độc lập** ⇒ chỉ cần duyệt `T` thông minh.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề khả thi:** tồn tại cách đạt nhóm `m` phần tử có AND chứa `T` ⇔ tổng `m` giá trị `cost(x, T)` nhỏ nhất `≤ k`. (Mỗi phần tử độc lập; với mỗi phần tử, tăng lên số nhỏ nhất chứa `T` là rẻ nhất.)

**Đơn điệu theo tập con:** nếu `T` khả thi thì mọi `T' ⊆ T` cũng khả thi (chứa `T` ⇒ chứa `T'`).

**Tham lam bit:** `ans = 0`; với `bit` từ cao xuống thấp, thử `T = ans | (1 << bit)`; nếu khả thi thì `ans = T`. Vì so sánh số nguyên là so sánh từ bit cao, và đơn điệu theo tập con bảo đảm các bit đã giữ không bao giờ phải bỏ, kết quả là đáp án lớn nhất.

Số bit cần xét: đáp án ≤ `max(nums) + k` ⇒ khoảng 31 bit. Mỗi lần kiểm tra `O(n log n)` ⇒ tổng `O(31 · n log n)`.

## 7. Các cách tiếp theo

- **Cách 2:** tham lam bit, nhưng chi phí tính bằng **duyệt** `y = x, x+1, …` (chỉ để kiểm chứng; chậm).
- **Cách 3:** tham lam bit + chi phí công thức + sắp xếp (lời giải nộp chính).
- **Cách 4:** như Cách 3 nhưng dùng `heapq.nsmallest` và **dừng sớm** khi tổng vượt `k`.

### 7.1. Cách 2 — Chi phí bằng duyệt

In [7]:
def greedy_with_brute_cost(nums, k, m):
    ans = 0
    for bit in range((max(nums) + k).bit_length() - 1, -1, -1):
        T = ans | (1 << bit)
        cost = sorted(min_cost_brute(x, T) for x in nums)
        if sum(cost[:m]) <= k:
            ans = T
    return ans

for nums, k, m, exp in examples:
    print(greedy_with_brute_cost(nums, k, m), "kỳ vọng", exp)

6 kỳ vọng 6
4 kỳ vọng 4
2 kỳ vọng 2


**Độ phức tạp:** chi phí duyệt có thể tới `~T` bước — không dùng được với `10^9`.

### 7.2. Cách 3 — Tham lam bit + công thức (lời giải chính)

In [8]:
def greedy_formula(nums, k, m):
    ans = 0
    for bit in range((max(nums) + k).bit_length() - 1, -1, -1):
        T = ans | (1 << bit)
        cost = []
        for x in nums:
            j = (T & ~x).bit_length()
            mask = (1 << j) - 1
            cost.append((T & mask) - (x & mask))
        cost.sort()
        if sum(cost[:m]) <= k:
            ans = T
    return ans

for nums, k, m, exp in examples:
    print(greedy_formula(nums, k, m), "kỳ vọng", exp)

6 kỳ vọng 6
4 kỳ vọng 4
2 kỳ vọng 2


**Độ phức tạp Cách 3:** `O(B · n log n)` với `B ≈ 31` ⇒ ~`2,5·10^7` phép so sánh khi `n = 5·10^4`; bộ nhớ `O(n)`.

### 7.3. Cách 4 — `heapq.nsmallest` + dừng sớm

`heapq.nsmallest(m, cost)` chạy `O(n log m)`; ngoài ra nếu `T` không đổi được bit nào khả thi thì phép kiểm tra chỉ cần tổng `m` giá trị nhỏ nhất.

In [9]:
def greedy_heap(nums, k, m):
    ans = 0
    for bit in range((max(nums) + k).bit_length() - 1, -1, -1):
        T = ans | (1 << bit)
        cost = [(T & ((1 << (T & ~x).bit_length()) - 1)) - (x & ((1 << (T & ~x).bit_length()) - 1)) for x in nums]
        if sum(heapq.nsmallest(m, cost)) <= k:
            ans = T
    return ans

for nums, k, m, exp in examples:
    print(greedy_heap(nums, k, m), "kỳ vọng", exp)

6 kỳ vọng 6
4 kỳ vọng 4
2 kỳ vọng 2


In [10]:
class SolutionHeap:
    def maximumAND(self, nums: List[int], k: int, m: int) -> int:
        return greedy_heap(nums, k, m)

print([SolutionHeap().maximumAND(a, k, m) for a, k, m, _ in examples])

[6, 4, 2]


**Độ phức tạp Cách 4:** `O(B · n log m)`; nhanh hơn khi `m` nhỏ.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 (`[1, 2, 8, 4]`, `k = 7`, `m = 3`).

In [11]:
def trace(nums, k, m):
    ans = 0
    for bit in range((max(nums) + k).bit_length() - 1, -1, -1):
        T = ans | (1 << bit)
        cost = [min_cost(x, T) for x in nums]
        s = sum(sorted(cost)[:m]); ok = s <= k
        print(f"bit {bit}: thử T={T:2} ({T:05b}) chi phí={cost} -> {m} rẻ nhất tổng {s} {'≤' if ok else '>'} {k} -> {'giữ' if ok else 'bỏ'}")
        if ok: ans = T
    print("đáp án:", ans)

trace([1, 2, 8, 4], 7, 3)

bit 3: thử T= 8 (01000) chi phí=[7, 6, 0, 4] -> 3 rẻ nhất tổng 10 > 7 -> bỏ
bit 2: thử T= 4 (00100) chi phí=[3, 2, 4, 0] -> 3 rẻ nhất tổng 5 ≤ 7 -> giữ
bit 1: thử T= 6 (00110) chi phí=[5, 4, 6, 2] -> 3 rẻ nhất tổng 11 > 7 -> bỏ
bit 0: thử T= 5 (00101) chi phí=[4, 3, 5, 1] -> 3 rẻ nhất tổng 8 > 7 -> bỏ
đáp án: 4


Bit 3 (`T = 8`) bị loại: số `8` sẵn có (chi phí 0) nhưng hai số rẻ tiếp theo là `4 → 8` (4) và `2 → 8` (6), tổng `10 > 7`. Bit 2 (`T = 4`) giữ được với tổng `0 + 2 + 3 = 5`. Bit 1 (`T = 6`) cần `2 + 4 + 5 = 11 > 7` nên bỏ; bit 0 (`T = 5`) cần `1 + 3 + 4 = 8 > 7` cũng bỏ. Đáp án `4`.

In [12]:
trace([3, 1, 2], 8, 2)

bit 3: thử T= 8 (01000) chi phí=[5, 7, 6] -> 2 rẻ nhất tổng 11 > 8 -> bỏ
bit 2: thử T= 4 (00100) chi phí=[1, 3, 2] -> 2 rẻ nhất tổng 3 ≤ 8 -> giữ
bit 1: thử T= 6 (00110) chi phí=[3, 5, 4] -> 2 rẻ nhất tổng 7 ≤ 8 -> giữ
bit 0: thử T= 7 (00111) chi phí=[4, 6, 5] -> 2 rẻ nhất tổng 9 > 8 -> bỏ
đáp án: 6


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def maximumAND(self, nums: List[int], k: int, m: int) -> int:
        ans = 0
        for bit in range((max(nums) + k).bit_length() - 1, -1, -1):
            T = ans | (1 << bit)
            cost = []
            for x in nums:
                j = (T & ~x).bit_length()        # bit cao nhất T có mà x thiếu (+1)
                mask = (1 << j) - 1
                cost.append((T & mask) - (x & mask))
            cost.sort()
            if sum(cost[:m]) <= k:
                ans = T
        return ans

sol = Solution()
for nums, k, m, exp in examples:
    got = sol.maximumAND(nums, k, m)
    print(nums, k, m, "->", got, "✔" if got == exp else "✘")

[3, 1, 2] 8 2 -> 6 ✔
[1, 2, 8, 4] 7 3 -> 4 ✔
[1, 1] 3 2 -> 2 ✔


## 10. Vì sao thuật toán đúng

1. **Công thức chi phí.** Gọi `p = j − 1` là bit cao nhất trong `T & ~x`. Mọi `y ≥ x` chứa `T`: xét bit cao nhất mà `y` khác `x`. Nếu nó cao hơn `p`, `y` lớn hơn `x` ít nhất ở bit đó ⇒ `y ≥ (x >> j << j) + 2^j > ứng viên của ta`. Nếu nó thấp hơn `p`, bit `p` của `y` vẫn là 0 ⇒ `y` không chứa `T`. Vậy `y` phải bật bit `p` và giữ các bit trên `p` như `x`; phần dưới `p` nhỏ nhất mà chứa `T` là đúng `T & (2^p − 1)`. Do đó `y = (x >> j << j) | (T & mask)`.
2. **Khả thi:** chi phí cho nhóm là tổng chi phí từng phần tử (độc lập), nên chọn `m` phần tử rẻ nhất là tối ưu.
3. **Đơn điệu & tham lam:** gọi `A*` là đáp án. Khi xét bit `b`, giả sử `ans` = phần của `A*` trên các bit cao hơn `b`. Nếu `A*` có bit `b` thì `T = ans | 2^b ⊆ A*` khả thi ⇒ được giữ. Nếu `A*` không có bit `b` mà `T` khả thi, thì có đáp án ≥ `T` > `A*` (vì `T` trùng `A*` ở các bit cao và lớn hơn tại `b`) — mâu thuẫn. Quy nạp ⇒ `ans = A*`.
4. **Đủ bit:** mọi giá trị sau khi tăng ≤ `max(nums) + k` ⇒ AND không vượt quá số đó ⇒ duyệt tới `bit_length(max + k) − 1` là đủ.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `B = bit_length(max(nums) + k) ≤ 31`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | hàm mũ | `O(n)` |
| 2. Tham lam + chi phí duyệt | `O(B · n · T)` | `O(n)` |
| 3. Tham lam + công thức + sort | `O(B · n log n)` | `O(n)` |
| 4. Tham lam + `nsmallest` | `O(B · n log m)` | `O(n)` |

## 12. Bộ test

In [14]:
def run_all(nums, k, m, with_brute=False):
    res = [greedy_formula(nums, k, m), greedy_heap(nums, k, m), Solution().maximumAND(nums, k, m)]
    if max(nums) + k <= 200:
        res.append(greedy_with_brute_cost(nums, k, m))
    if with_brute:
        res.append(brute(nums, k, m))
    return res

for nums, k, m, exp in examples:
    assert all(v == exp for v in run_all(nums, k, m))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [15]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([5], 0 + 1, 1, 6),            # một phần tử: tăng tối đa -> 6
    ([7, 7], 1, 2, 7),             # đã bằng nhau
    ([1, 2], 1, 1, 3),             # m = 1: lấy max(x) + k
    ([4, 3], 1, 2, 4),             # 3 -> 4
    ([1, 1, 1], 2, 3, 1),          # không đủ để đưa cả ba lên 2
]
for nums, k, m, exp in cases:
    r = run_all(nums, k, m, with_brute=True)
    assert all(v == exp for v in r), (nums, k, m, r)
print("Ca biên: OK")

Ca biên: OK


In [16]:
# 12.3 phản ví dụ: tham lam "chọn m số lớn nhất rồi dồn k vào" không tối ưu
def wrong_largest(nums, k, m):
    chosen = sorted(nums)[-m:]
    return Solution().maximumAND(chosen, k, m)

rng = random.Random(1)
for _ in range(5000):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(2, 5))]; m = rng.randint(1, len(nums)); k = rng.randint(1, 10)
    if wrong_largest(nums, k, m) != Solution().maximumAND(nums, k, m):
        print("phản ví dụ:", nums, k, m, "| chọn m số lớn nhất:", wrong_largest(nums, k, m), "| đúng:", Solution().maximumAND(nums, k, m)); break

phản ví dụ: [18, 8, 12, 8, 8] 5 4 | chọn m số lớn nhất: 1 | đúng: 9


In [17]:
# 12.4 đối chiếu ngẫu nhiên (vét cạn trên dữ liệu rất nhỏ)
rng = random.Random(3806)
for _ in range(150):
    n = rng.randint(1, 3); nums = [rng.randint(1, 6) for _ in range(n)]
    k = rng.randint(1, 5); m = rng.randint(1, n)
    r = run_all(nums, k, m, with_brute=True)
    assert len(set(r)) == 1, (nums, k, m, r)
for _ in range(300):
    nums = [rng.randint(1, 60) for _ in range(rng.randint(1, 12))]; k = rng.randint(1, 100); m = rng.randint(1, len(nums))
    r = run_all(nums, k, m)
    assert len(set(r)) == 1, (nums, k, m, r)
print("450 test ngẫu nhiên: OK")

450 test ngẫu nhiên: OK


In [18]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
nums = [rng.randint(1, 10**9) for _ in range(5 * 10**4)]
for k, m in ((10**9, 1), (10**9, 100), (10**9, 5 * 10**4)):
    t0 = time.time(); a = Solution().maximumAND(nums, k, m); t1 = time.time()
    print(f"k={k} m={m}: {a}, khớp Cách 4: {a == greedy_heap(nums, k, m)}, {t1 - t0:.2f}s")

k=1000000000 m=1: 1999998593, khớp Cách 4: True, 0.67s


k=1000000000 m=100: 1008983995, khớp Cách 4: True, 0.53s


k=1000000000 m=50000: 70924, khớp Cách 4: True, 0.47s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chi phí = `T − x` khi `x < T` (coi phải tăng lên đúng `T`)

Số nhỏ nhất chứa `T` có thể **lớn hơn `T`** (khi `x` đã có bit cao hơn) hoặc **nhỏ hơn** việc tăng lên `T` (khi `x > T`). Công thức `max(0, T − x)` sai cả hai chiều.

In [19]:
def wrong_cost(x, T):
    return max(0, T - x)                       # SAI

for x, T in [(8, 4), (13, 10), (5, 6)]:
    print(f"x={x} T={T}: sai {wrong_cost(x, T)} | đúng {min_cost(x, T)}")

x=8 T=4: sai 0 | đúng 4
x=13 T=10: sai 0 | đúng 1
x=5 T=6: sai 1 | đúng 1


**Vì sao sai:** `x = 8`, `T = 4`: `8 = 1000` không có bit 2; số nhỏ nhất ≥ 8 chứa bit 2 là `12`, chi phí 4 (không phải 0).

### Lỗi 2 — Bắt đầu từ bit quá thấp (ví dụ `bit_length(max(nums)) − 1`)

Sau khi tăng, giá trị có thể vượt `max(nums)` và có bit cao hơn. Phải dùng `max(nums) + k`.

In [20]:
def wrong_start_bit(nums, k, m):
    ans = 0
    for bit in range(max(nums).bit_length() - 1, -1, -1):        # SAI
        T = ans | (1 << bit)
        if sum(sorted(min_cost(x, T) for x in nums)[:m]) <= k: ans = T
    return ans

print("sai:", wrong_start_bit([1, 1], 3, 2), "| đúng:", Solution().maximumAND([1, 1], 3, 2))

sai: 1 | đúng: 2


### Lỗi 3 — Thử từng bit **độc lập** (không cộng dồn `ans`)

Mỗi bit khả thi riêng lẻ không có nghĩa là **cùng nhau** khả thi; phải thử `ans | bit`.

In [21]:
def wrong_independent(nums, k, m):
    ans = 0
    for bit in range((max(nums) + k).bit_length() - 1, -1, -1):
        T = 1 << bit                                               # SAI: không giữ các bit đã chọn
        if sum(sorted(min_cost(x, T) for x in nums)[:m]) <= k: ans |= T
    return ans

print("sai:", wrong_independent([3, 1, 2], 8, 2), "| đúng:", Solution().maximumAND([3, 1, 2], 8, 2))

sai: 7 | đúng: 6


### Lỗi 4 — Tràn `1 << j` / dùng `~x` với số không dấu (ngôn ngữ khác)

Trong C++/Java, `~x` trên kiểu không dấu hoặc `1 << 31` với `int` gây tràn. Python không gặp, nhưng khi chuyển ngôn ngữ cần `long long` và `1LL << j`.

In [22]:
x, T = 2**30 + 5, 2**30 + 2**29
print("j =", (T & ~x).bit_length(), "| chi phí =", min_cost(x, T), "| kiểm tra trực tiếp:", ((x >> 30 << 30) | (T & (2**30 - 1))) - x)

j = 30 | chi phí = 536870907 | kiểm tra trực tiếp: 536870907


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tối đa hóa AND / OR"** ⇒ dựng đáp án theo bit từ cao xuống, kiểm tra khả thi với `ans | bit` (LC 421, LC 2935…).
- **"Số nhỏ nhất ≥ x chứa mặt nạ T"** ⇒ tìm bit cao nhất thiếu, thay phần thấp bằng `T`.
- **Chọn nhóm `m` phần tử với chi phí độc lập** ⇒ lấy `m` chi phí nhỏ nhất.
- Kiểm tra khả thi có tính **đơn điệu theo tập con bit** ⇒ tham lam bit đúng.

## 15. Biến thể của bài

1. **Tối đa hóa OR** của nhóm `m` phần tử sau tăng: không còn tách từng phần tử theo `T` (OR cần "ít nhất một" phần tử có bit) — khó hơn.
2. **Được tăng hoặc giảm**: chi phí `min(|y − x|)` với `y ⊇ T` — cần xét cả số lớn nhất `≤ x` chứa `T`.
3. **Mỗi phần tử có giới hạn tăng riêng** `limit[i]`: loại phần tử có chi phí vượt giới hạn.
4. **Toàn bộ mảng** (`m = n`): tổng mọi chi phí.

In [23]:
# Biến thể 2: được tăng HOẶC giảm (giá trị không âm). Chi phí = khoảng cách tới số gần nhất chứa T.
def nearest_cost(x, T):
    up = min_cost(x, T)
    best_down = None
    y = x
    while y >= 0 and best_down is None:          # duyệt (đủ dùng cho dữ liệu nhỏ)
        if y & T == T: best_down = x - y
        y -= 1
    return up if best_down is None else min(up, best_down)

def max_and_both(nums, k, m):
    ans = 0
    for bit in range((max(nums) + k).bit_length() - 1, -1, -1):
        T = ans | (1 << bit)
        if sum(sorted(nearest_cost(x, T) for x in nums)[:m]) <= k: ans = T
    return ans

print(max_and_both([3, 1, 2], 8, 2), max_and_both([8, 7], 1, 2), "(8 và 7 với k = 1: tăng 7 -> 8 cho AND 8)")
rng = random.Random(6)
for _ in range(3000):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(2, 4))]; k = rng.randint(1, 6); m = rng.randint(2, len(nums))
    if max_and_both(nums, k, m) != Solution().maximumAND(nums, k, m):
        print("ví dụ cho phép giảm có lợi:", nums, k, m, "| tăng/giảm:", max_and_both(nums, k, m), "| chỉ tăng:", Solution().maximumAND(nums, k, m)); break

6 8 (8 và 7 với k = 1: tăng 7 -> 8 cho AND 8)
ví dụ cho phép giảm có lợi: [12, 11, 25, 1] 3 3 | tăng/giảm: 11 | chỉ tăng: 10


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi mức tăng | hàm mũ | `O(n)` | Kiểm thử |
| 2. Chi phí duyệt | Tham lam + duyệt `y` | `O(B n T)` | `O(n)` | Kiểm chứng công thức |
| 3. Công thức + sort | Tham lam bit | `O(B n log n)` | `O(n)` | **Lời giải chuẩn** |
| 4. Công thức + `nsmallest` | Chọn `m` nhỏ nhất | `O(B n log m)` | `O(n)` | `m` nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maximumAND([5], 3, 1)` = ?

**Bài 2 (tính tay).** `min_cost(x, T)` với `x = 0b10011`, `T = 0b01100`.

**Bài 3 (điền TODO).** Viết `smallest_superset_ge(x, T)` trả về `y` (không phải chi phí).

**Bài 4 (tự viết lại).** Viết lại Cách 3 với `numpy`-free nhưng **không** sắp xếp toàn bộ: dùng `sorted` chỉ khi cần (gợi ý: `heapq.nsmallest`) và so sánh tốc độ.

**Bài 5 (mini challenge).** Ngoài AND lớn nhất, trả về **tập chỉ số** đạt được và **giá trị sau khi tăng** của chúng.

In [24]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def smallest_superset_ge(x, T):
    # TODO: j = (T & ~x).bit_length(); y = (x >> j << j) | (T & ((1 << j) - 1))
    pass

In [25]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def construct(nums, k, m):
    # TODO: tìm ans bằng Solution; sắp chỉ số theo chi phí với T = ans; lấy m chỉ số đầu
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `m = 1`. (2) Dồn hết vào một phần tử: `5 + 3 = 8`. (3) Đáp án `8`.

**Bài 2.** (1) `T & ~x = 0b01100 & 0b01100 = 0b01100` ⇒ `j = 4`. (2) `mask = 0b1111`; `T & mask = 12`, `x & mask = 3`. (3) Chi phí `9` (`y = 0b11100 = 28`).

**Bài 3.** (1) Nếu `x ⊇ T` thì `j = 0` ⇒ `y = x`. (2) Ngược lại theo công thức. (3) So với duyệt.

**Bài 4.** (1) `heapq.nsmallest(m, cost)`. (2) Đo thời gian với `m = 10` và `m = n`. (3) Kết quả giống nhau.

**Bài 5.** (1) Với `T = ans`, tính chi phí từng phần tử. (2) Chọn `m` chỉ số rẻ nhất. (3) Giá trị mới `x + cost`, kiểm tra AND ≥ ans và tổng chi phí ≤ `k`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().maximumAND([5], 3, 1) == 8
assert min_cost(0b10011, 0b01100) == 9 == min_cost_brute(0b10011, 0b01100)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [27]:
# Bài 3
def smallest_superset_ge_sol(x, T):
    j = (T & ~x).bit_length()
    return (x >> j << j) | (T & ((1 << j) - 1))

rng = random.Random(3)
for _ in range(2000):
    x, T = rng.randint(0, 300), rng.randint(0, 300)
    y = x
    while y & T != T: y += 1
    assert smallest_superset_ge_sol(x, T) == y
print("Bài 3 OK")

Bài 3 OK


In [28]:
# Bài 4: so sánh tốc độ sort vs nsmallest
rng = random.Random(4)
nums = [rng.randint(1, 10**9) for _ in range(2 * 10**4)]
for m in (10, 2 * 10**4):
    t0 = time.time(); a = greedy_formula(nums, 10**9, m); t1 = time.time(); b = greedy_heap(nums, 10**9, m); t2 = time.time()
    print(f"m={m}: sort {t1 - t0:.2f}s, nsmallest {t2 - t1:.2f}s, cùng kết quả: {a == b}")

m=10: sort 0.25s, nsmallest 0.18s, cùng kết quả: True


m=20000: sort 0.18s, nsmallest 0.24s, cùng kết quả: True


In [29]:
# Bài 5: dựng nhóm cụ thể
def construct_sol(nums, k, m):
    ans = Solution().maximumAND(nums, k, m)
    order = sorted(range(len(nums)), key=lambda i: min_cost(nums[i], ans))[:m]
    new_vals = [nums[i] + min_cost(nums[i], ans) for i in order]
    return ans, order, new_vals

for nums, k, m, _ in examples:
    ans, idx, vals = construct_sol(nums, k, m)
    a = vals[0]
    for v in vals[1:]: a &= v
    assert a & ans == ans and sum(v - nums[i] for i, v in zip(idx, vals)) <= k
    print(ans, idx, vals, "AND thực tế =", a)
print("Bài 5 OK")

6 [0, 2] [6, 6] AND thực tế = 6
4 [3, 1, 0] [4, 4, 4] AND thực tế = 4
2 [0, 1] [2, 2] AND thực tế = 2
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = 0
for bit in range((max(nums) + k).bit_length() - 1, -1, -1):
    T = ans | (1 << bit)
    cost = [(T & mask(x)) - (x & mask(x)) for x in nums]   # mask(x) = (1 << (T & ~x).bit_length()) - 1
    if sum(sorted(cost)[:m]) <= k:
        ans = T
return ans
```

**Checklist:**

- [ ] Bắt đầu từ bit cao nhất của `max(nums) + k`?
- [ ] Thử `ans | (1 << bit)` (cộng dồn), không thử bit riêng lẻ?
- [ ] Chi phí là "số nhỏ nhất ≥ x chứa T" trừ `x` (không phải `max(0, T − x)`)?
- [ ] Lấy `m` chi phí **nhỏ nhất**?
- [ ] Phần tử đã chứa `T` có chi phí 0?